# P7513 Cellpose Area Cutoff Explorer

This notebook loads the P7513 MERSCOPE SpatialData output, scans for a 250 um crop enriched for the smallest Cellpose masks, labels those tiny masks by area, and plots a high-bin histogram of Cellpose mask areas. Use `AREA_MODE = "um2"` for compact biological area labels, or `AREA_MODE = "px"` for pixel-equivalent values that map back to mask filtering thresholds.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatialdata as sd
from matplotlib import patheffects as path_effects
from scipy.spatial import cKDTree
from shapely.geometry import box as shapely_box

candidate_roots = [Path.cwd(), Path.cwd().parent, Path("/home/becalia/code/MerXen")]
REPO_ROOT = next((p for p in candidate_roots if (p / "src" / "merxen").exists()), candidate_roots[-1])
src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.visualization.sanity_plots import (  # noqa: E402
    SANITY_CROP_SIZE_UM,
    _build_pair_sanity_plans,
    _crop_single_shape,
    _get_background_image_crop,
    _resolve_dataset_mask_affine,
)

plt.rcParams["figure.dpi"] = 130

## Settings

In [ ]:
PAIR_ID = "P7513"
DATASET_NAME = "MERSCOPE"
PLATFORM = "merscope"

ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / PLATFORM / "latest" / "latest_spatialdata.zarr"
XENIUM_ZARR_PATH = REPO_ROOT / "results" / PAIR_ID / "xenium" / "latest" / "latest_spatialdata.zarr"
OUTPUT_DIR = REPO_ROOT / "results" / PAIR_ID / "notebook_outputs"

CELLPOSE_SHAPE_KEY = "MOSAIK_cellpose"
PROSEG_SHAPE_KEY = "MOSAIK_proseg"

# Default behavior: scan for a 250 um crop enriched for very small Cellpose masks.
AUTO_SCAN_TINY_MASK_CROP = True
CROP_SIZE_UM = 250.0
TINY_MASK_PERCENTILE = 1.0

# Optional: ignore scan windows with too many total Cellpose masks.
# Leave as None to purely maximize tiny-mask enrichment.
SCAN_MAX_TOTAL_MASKS = None

# Manual fallback crop, format: xmin, ymin, xmax, ymax.
CROP_BBOX = (5500.0, 5230.0, 5750.0, 5480.0)

# Set True to use the production 250 um sanity crop instead of the smaller zoom.
USE_PRODUCTION_SANITY_CROP = False

# "um2" gives compact labels; "px" gives pixel-equivalent mask sizes.
AREA_MODE = "um2"

# Optional cutoff to visualize. Set to a number in the current AREA_MODE.
AREA_CUTOFF = None

SHOW_BACKGROUND = True
# "all" labels every mask above LOW_AREA_LABEL_CAP_PERCENTILE, then samples
# labels below that percentile so tiny masks do not bury the plot.
# Other options: "tiny_cellpose" or "none".
LABEL_MODE = "all"
HIGHLIGHT_TINY_MASKS = True
LOW_AREA_LABEL_CAP_PERCENTILE = 5.0
MAX_LOW_AREA_LABELS = 80
LABEL_FONTSIZE = 7.0
HIST_BINS = 300
HIST_MAX_PERCENTILE = 99.5
HIST_LOG_Y = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO_ROOT}")
print(f"SpatialData: {ZARR_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

## Load Data

In [ ]:
if not ZARR_PATH.exists():
    raise FileNotFoundError(f"Missing P7513 MERSCOPE zarr: {ZARR_PATH}")

sdata = sd.read_zarr(ZARR_PATH)
print("Images:", list(sdata.images.keys()))
print("Shapes:", list(sdata.shapes.keys()))
print("Tables:", list(sdata.tables.keys()))

missing = [key for key in [CELLPOSE_SHAPE_KEY, PROSEG_SHAPE_KEY] if key not in sdata.shapes]
if missing:
    raise KeyError(f"Missing expected mask shape layer(s): {missing}")

x_transform, y_transform = _resolve_dataset_mask_affine(DATASET_NAME, zarr_path=ZARR_PATH)
PIXEL_AREA_UM2 = abs((x_transform[0] * y_transform[1]) - (x_transform[1] * y_transform[0]))
print(f"Pixel area in SpatialData units: {PIXEL_AREA_UM2:.6g} um2")

In [ ]:
def _cellpose_scan_frame(sdata_obj, shape_key: str) -> pd.DataFrame:
    gdf = sdata_obj.shapes[shape_key]
    label_points = gdf.geometry.representative_point()
    frame = pd.DataFrame(
        {
            "shape_index": gdf.index.to_numpy(),
            "x": label_points.x.to_numpy(dtype=float),
            "y": label_points.y.to_numpy(dtype=float),
            "area_um2": gdf.geometry.area.to_numpy(dtype=float),
        }
    )
    frame = frame[
        np.isfinite(frame["x"])
        & np.isfinite(frame["y"])
        & np.isfinite(frame["area_um2"])
        & (frame["area_um2"] > 0)
    ].reset_index(drop=True)
    frame["area_px"] = frame["area_um2"] / PIXEL_AREA_UM2
    return frame


def scan_for_tiny_mask_crop(
    sdata_obj,
    shape_key: str,
    *,
    crop_size_um: float,
    tiny_percentile: float,
    max_total_masks: int | None = None,
) -> tuple[tuple[float, float, float, float], pd.DataFrame, dict[str, float]]:
    frame = _cellpose_scan_frame(sdata_obj, shape_key)
    tiny_cutoff_um2 = float(np.percentile(frame["area_um2"], tiny_percentile))
    tiny_cutoff_px = tiny_cutoff_um2 / PIXEL_AREA_UM2
    tiny = frame[frame["area_um2"] <= tiny_cutoff_um2].copy()
    if tiny.empty:
        raise RuntimeError("No tiny Cellpose masks found for scan.")

    half = float(crop_size_um) / 2.0
    bounds = sdata_obj.shapes[shape_key].total_bounds.astype(float)
    minx, miny, maxx, maxy = bounds
    if (maxx - minx) < crop_size_um or (maxy - miny) < crop_size_um:
        raise ValueError("Requested crop is larger than the Cellpose field of view.")

    candidates = tiny[["x", "y"]].to_numpy(dtype=float)
    candidates[:, 0] = np.clip(candidates[:, 0], minx + half, maxx - half)
    candidates[:, 1] = np.clip(candidates[:, 1], miny + half, maxy - half)
    candidates = np.unique(np.round(candidates, 3), axis=0)

    all_xy = frame[["x", "y"]].to_numpy(dtype=float)
    tiny_xy = tiny[["x", "y"]].to_numpy(dtype=float)
    all_tree = cKDTree(all_xy)
    tiny_tree = cKDTree(tiny_xy)

    tiny_counts = tiny_tree.query_ball_point(
        candidates,
        r=half,
        p=np.inf,
        return_length=True,
    )
    total_counts = all_tree.query_ball_point(
        candidates,
        r=half,
        p=np.inf,
        return_length=True,
    )
    tiny_fraction = tiny_counts / np.maximum(total_counts, 1)
    eligible = np.ones(len(candidates), dtype=bool)
    if max_total_masks is not None:
        eligible &= total_counts <= int(max_total_masks)
    if not eligible.any():
        eligible = np.ones(len(candidates), dtype=bool)

    eligible_indices = np.flatnonzero(eligible)
    best_idx = max(
        eligible_indices,
        key=lambda i: (int(tiny_counts[i]), float(tiny_fraction[i]), -int(total_counts[i])),
    )
    cx, cy = candidates[best_idx]
    bbox = (float(cx - half), float(cy - half), float(cx + half), float(cy + half))

    scan_results = pd.DataFrame(
        {
            "center_x": candidates[:, 0],
            "center_y": candidates[:, 1],
            "tiny_mask_count": tiny_counts,
            "total_mask_count": total_counts,
            "tiny_mask_fraction": tiny_fraction,
        }
    ).sort_values(
        ["tiny_mask_count", "tiny_mask_fraction", "total_mask_count"],
        ascending=[False, False, True],
    )
    scan_summary = {
        "tiny_cutoff_um2": tiny_cutoff_um2,
        "tiny_cutoff_px": tiny_cutoff_px,
        "chosen_tiny_count": float(tiny_counts[best_idx]),
        "chosen_total_count": float(total_counts[best_idx]),
        "chosen_tiny_fraction": float(tiny_fraction[best_idx]),
    }
    return bbox, scan_results, scan_summary


scan_results = pd.DataFrame()
scan_summary = {}
if AUTO_SCAN_TINY_MASK_CROP:
    crop_bbox, scan_results, scan_summary = scan_for_tiny_mask_crop(
        sdata,
        CELLPOSE_SHAPE_KEY,
        crop_size_um=CROP_SIZE_UM,
        tiny_percentile=TINY_MASK_PERCENTILE,
        max_total_masks=SCAN_MAX_TOTAL_MASKS,
    )
elif USE_PRODUCTION_SANITY_CROP:
    if not XENIUM_ZARR_PATH.exists():
        raise FileNotFoundError(f"Missing P7513 Xenium zarr: {XENIUM_ZARR_PATH}")
    xenium_sdata = sd.read_zarr(XENIUM_ZARR_PATH)
    crop_bbox = _build_pair_sanity_plans(
        sdata,
        xenium_sdata,
        crop_size_um=SANITY_CROP_SIZE_UM,
    )[0].display_bbox
else:
    crop_bbox = tuple(map(float, CROP_BBOX))

x0, y0, x1, y1 = crop_bbox
print(f"Crop bbox: x=({x0:.1f}, {x1:.1f}), y=({y0:.1f}, {y1:.1f})")
print(f"Crop size: {x1 - x0:.1f} x {y1 - y0:.1f}")
if scan_summary:
    print(
        f"Tiny cutoff p{TINY_MASK_PERCENTILE:g}: "
        f"{scan_summary['tiny_cutoff_um2']:.3g} um2 / "
        f"{scan_summary['tiny_cutoff_px']:.1f} px-equivalent"
    )
    print(
        "Chosen window: "
        f"{scan_summary['chosen_tiny_count']:.0f} tiny masks, "
        f"{scan_summary['chosen_total_count']:.0f} total masks, "
        f"fraction={scan_summary['chosen_tiny_fraction']:.3f}"
    )
scan_results.head(10)


## Helper Functions

In [ ]:
def _area_mode_col() -> str:
    if AREA_MODE == "um2":
        return "area_um2"
    if AREA_MODE == "px":
        return "area_px"
    raise ValueError('AREA_MODE must be "um2" or "px"')


def _area_axis_label() -> str:
    return "area (um2)" if AREA_MODE == "um2" else "area (px-equivalent)"


def _add_area_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["area_px"] = df["area_um2"] / PIXEL_AREA_UM2
    return df


def _format_area_label(value: float) -> str:
    value = float(value)
    if value < 1:
        return f"{value:.2g}"
    if value < 10:
        return f"{value:.1f}"
    return f"{value:.0f}"


def shape_area_table(sdata_obj, shape_key: str) -> pd.DataFrame:
    gdf = sdata_obj.shapes[shape_key]
    area_um2 = gdf.geometry.area.to_numpy(dtype=float)
    out = pd.DataFrame(
        {
            "shape_index": gdf.index.to_numpy(),
            "area_um2": area_um2,
        }
    )
    out = out[np.isfinite(out["area_um2"]) & (out["area_um2"] > 0)].reset_index(drop=True)
    return _add_area_columns(out)


def crop_shapes_with_areas(sdata_obj, shape_key: str, bbox) -> pd.DataFrame:
    gdf = _crop_single_shape(sdata_obj, shape_key, bbox).copy()
    if len(gdf) == 0:
        return pd.DataFrame(
            columns=["shape_index", "geometry", "area_um2", "area_px", "label_x", "label_y"]
        )

    crop_poly = shapely_box(*bbox)
    visible = gdf.geometry.intersection(crop_poly)
    keep = visible.notna() & ~visible.is_empty
    gdf = gdf.loc[keep].copy()
    visible = visible.loc[keep]
    label_points = visible.representative_point()

    gdf["shape_index"] = gdf.index.to_numpy()
    gdf["area_um2"] = gdf.geometry.area.to_numpy(dtype=float)
    gdf["label_x"] = label_points.x.to_numpy(dtype=float)
    gdf["label_y"] = label_points.y.to_numpy(dtype=float)
    return _add_area_columns(gdf)


def draw_mask_panel(
    ax,
    gdf,
    title: str,
    color: str,
    bbox,
    *,
    bg=None,
    cutoff=None,
    label_all: bool = True,
    label_area_max: float | None = None,
    low_area_label_cap: float | None = None,
) -> None:
    if bg is not None:
        ax.imshow(
            bg["rgb"],
            extent=bg["extent_um"],
            origin="lower",
            interpolation="nearest",
            alpha=0.85,
        )

    if len(gdf) > 0:
        area_col = _area_mode_col()
        if cutoff is None:
            gdf.plot(ax=ax, color=color, alpha=0.20, linewidth=0, zorder=3)
            gdf.boundary.plot(ax=ax, color=color, linewidth=1.2, alpha=0.95, zorder=4)
        else:
            below = gdf[area_col] < float(cutoff)
            if (~below).any():
                gdf.loc[~below].plot(ax=ax, color=color, alpha=0.20, linewidth=0, zorder=3)
                gdf.loc[~below].boundary.plot(ax=ax, color=color, linewidth=1.2, alpha=0.95, zorder=4)
            if below.any():
                gdf.loc[below].plot(ax=ax, color="#ef4444", alpha=0.28, linewidth=0, zorder=5)
                gdf.loc[below].boundary.plot(ax=ax, color="#ef4444", linewidth=1.6, alpha=1.0, zorder=6)

        text_effects = [
            path_effects.Stroke(linewidth=2.4, foreground="black"),
            path_effects.Normal(),
        ]
        if label_area_max is not None:
            label_gdf = gdf[gdf[area_col] <= float(label_area_max)]
        elif label_all:
            label_gdf = gdf
        else:
            label_gdf = gdf.iloc[0:0]
        if (
            low_area_label_cap is not None
            and MAX_LOW_AREA_LABELS is not None
            and len(label_gdf) > 0
        ):
            low_label_gdf = label_gdf[label_gdf[area_col] <= float(low_area_label_cap)]
            high_label_gdf = label_gdf[label_gdf[area_col] > float(low_area_label_cap)]
            if len(low_label_gdf) > int(MAX_LOW_AREA_LABELS):
                low_label_gdf = low_label_gdf.sample(n=int(MAX_LOW_AREA_LABELS), random_state=42)
            label_gdf = pd.concat([high_label_gdf, low_label_gdf], axis=0)

        for row in label_gdf.itertuples():
            value = getattr(row, area_col)
            ax.text(
                row.label_x,
                row.label_y,
                _format_area_label(value),
                color="white",
                fontsize=LABEL_FONTSIZE,
                ha="center",
                va="center",
                path_effects=text_effects,
                zorder=20,
            )

    x0, y0, x1, y1 = bbox
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")


## Area Tables

In [ ]:
cellpose_areas = shape_area_table(sdata, CELLPOSE_SHAPE_KEY)
tiny_cutoff_um2 = float(
    scan_summary.get(
        "tiny_cutoff_um2",
        np.percentile(cellpose_areas["area_um2"].to_numpy(dtype=float), TINY_MASK_PERCENTILE),
    )
)
tiny_cutoff_px = tiny_cutoff_um2 / PIXEL_AREA_UM2
tiny_cutoff_current_mode = tiny_cutoff_um2 if AREA_MODE == "um2" else tiny_cutoff_px
low_area_label_cap_um2 = float(
    np.percentile(
        cellpose_areas["area_um2"].to_numpy(dtype=float),
        LOW_AREA_LABEL_CAP_PERCENTILE,
    )
)
low_area_label_cap_px = low_area_label_cap_um2 / PIXEL_AREA_UM2
low_area_label_cap_current_mode = low_area_label_cap_um2 if AREA_MODE == "um2" else low_area_label_cap_px
display_cutoff = AREA_CUTOFF
if display_cutoff is None and HIGHLIGHT_TINY_MASKS:
    display_cutoff = tiny_cutoff_current_mode

cellpose_crop = crop_shapes_with_areas(sdata, CELLPOSE_SHAPE_KEY, crop_bbox)
proseg_crop = crop_shapes_with_areas(sdata, PROSEG_SHAPE_KEY, crop_bbox)


def _n_tiny(df: pd.DataFrame) -> int:
    return int((df["area_um2"] <= tiny_cutoff_um2).sum())

summary = pd.DataFrame(
    [
        {
            "layer": "Cellpose crop",
            "n_masks": len(cellpose_crop),
            "median_area_um2": cellpose_crop["area_um2"].median(),
            "median_area_px": cellpose_crop["area_px"].median(),
            "n_tiny_masks": _n_tiny(cellpose_crop),
        },
        {
            "layer": "ProSeg crop",
            "n_masks": len(proseg_crop),
            "median_area_um2": proseg_crop["area_um2"].median(),
            "median_area_px": proseg_crop["area_px"].median(),
            "n_tiny_masks": _n_tiny(proseg_crop),
        },
        {
            "layer": "Cellpose full field",
            "n_masks": len(cellpose_areas),
            "median_area_um2": cellpose_areas["area_um2"].median(),
            "median_area_px": cellpose_areas["area_px"].median(),
            "n_tiny_masks": _n_tiny(cellpose_areas),
        },
    ]
)
summary

## Zoomed Mask Comparison

In [ ]:
bg = None
if SHOW_BACKGROUND:
    bg = _get_background_image_crop(sdata, DATASET_NAME, crop_bbox, zarr_path=ZARR_PATH)

label_mode = LABEL_MODE.lower()
if label_mode not in {"tiny_cellpose", "all", "none"}:
    raise ValueError('LABEL_MODE must be "tiny_cellpose", "all", or "none"')

cellpose_label_all = label_mode == "all"
cellpose_label_area_max = tiny_cutoff_current_mode if label_mode == "tiny_cellpose" else None
proseg_label_all = label_mode == "all"
label_cap_threshold = low_area_label_cap_current_mode if label_mode == "all" else None

fig, axes = plt.subplots(2, 2, figsize=(13, 12), constrained_layout=True)
draw_mask_panel(
    axes[0, 0],
    cellpose_crop,
    f"Cellpose masks ({len(cellpose_crop)}; tiny={_n_tiny(cellpose_crop)}) - labels show {_area_axis_label()}",
    "#a855f7",
    crop_bbox,
    bg=bg,
    cutoff=display_cutoff,
    label_all=cellpose_label_all,
    label_area_max=cellpose_label_area_max,
    low_area_label_cap=label_cap_threshold,
)
draw_mask_panel(
    axes[0, 1],
    proseg_crop,
    f"ProSeg masks ({len(proseg_crop)}) - labels show {_area_axis_label()}",
    "#22c55e",
    crop_bbox,
    bg=bg,
    label_all=proseg_label_all,
    low_area_label_cap=label_cap_threshold,
)
draw_mask_panel(
    axes[1, 0],
    cellpose_crop,
    f"Cellpose masks ({len(cellpose_crop)}) - no labels",
    "#a855f7",
    crop_bbox,
    bg=bg,
    cutoff=display_cutoff,
    label_all=False,
)
draw_mask_panel(
    axes[1, 1],
    proseg_crop,
    f"ProSeg masks ({len(proseg_crop)}) - no labels",
    "#22c55e",
    crop_bbox,
    bg=bg,
    label_all=False,
)

mask_plot_path = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_cellpose_proseg_area_crop.png"
fig.savefig(mask_plot_path, dpi=220, bbox_inches="tight")
print(f"Saved {mask_plot_path}")
plt.show()

## Cellpose Area Histogram

In [ ]:
area_col = _area_mode_col()
areas = cellpose_areas[area_col].to_numpy(dtype=float)
areas = areas[np.isfinite(areas) & (areas > 0)]
hist_max = np.percentile(areas, HIST_MAX_PERCENTILE) if HIST_MAX_PERCENTILE < 100 else areas.max()
hist_values = areas[areas <= hist_max]

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.hist(hist_values, bins=HIST_BINS, color="#7c3aed", alpha=0.78)

for pct, color in [(1, "#f97316"), (5, "#ef4444"), (10, "#0ea5e9"), (50, "#111827")]:
    value = np.percentile(areas, pct)
    if value <= hist_max:
        ax.axvline(value, color=color, linewidth=1.4, linestyle="--", label=f"p{pct}: {value:.1f}")

if AREA_CUTOFF is not None:
    ax.axvline(float(AREA_CUTOFF), color="#dc2626", linewidth=2.5, label=f"cutoff: {AREA_CUTOFF:g}")

ax.set_title(f"P7513 Cellpose mask areas, clipped at p{HIST_MAX_PERCENTILE}")
ax.set_xlabel(_area_axis_label())
ax.set_ylabel("mask count")
if HIST_LOG_Y:
    ax.set_yscale("log")
ax.legend(frameon=False, fontsize=8)

hist_path = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_cellpose_area_histogram.png"
fig.savefig(hist_path, dpi=220, bbox_inches="tight")
print(f"Saved {hist_path}")
plt.show()

## Large Mask Scan

In [ ]:
BIG_MASK_PERCENTILE = 99.0
BIG_MASK_MIN_AREA_UM2 = 300.0
BIG_CROP_SIZE_UM = 250.0
BIG_SCAN_MAX_TOTAL_MASKS = None
BIG_HIST_BINS = 120


In [ ]:
def scan_for_big_mask_crop(
    sdata_obj,
    shape_key: str,
    *,
    crop_size_um: float,
    big_percentile: float,
    big_min_area_um2: float | None = None,
    max_total_masks: int | None = None,
) -> tuple[tuple[float, float, float, float], pd.DataFrame, dict[str, float]]:
    frame = _cellpose_scan_frame(sdata_obj, shape_key)
    big_cutoff_um2 = (
        float(big_min_area_um2)
        if big_min_area_um2 is not None
        else float(np.percentile(frame["area_um2"], big_percentile))
    )
    big_cutoff_px = big_cutoff_um2 / PIXEL_AREA_UM2
    big = frame[frame["area_um2"] >= big_cutoff_um2].copy()
    if big.empty:
        raise RuntimeError(f"No Cellpose masks found with area >= {big_cutoff_um2:g} um2.")

    half = float(crop_size_um) / 2.0
    minx, miny, maxx, maxy = sdata_obj.shapes[shape_key].total_bounds.astype(float)
    candidates = big[["x", "y"]].to_numpy(dtype=float)
    candidates[:, 0] = np.clip(candidates[:, 0], minx + half, maxx - half)
    candidates[:, 1] = np.clip(candidates[:, 1], miny + half, maxy - half)
    candidates = np.unique(np.round(candidates, 3), axis=0)

    all_xy = frame[["x", "y"]].to_numpy(dtype=float)
    big_xy = big[["x", "y"]].to_numpy(dtype=float)
    all_tree = cKDTree(all_xy)
    big_tree = cKDTree(big_xy)
    big_counts = big_tree.query_ball_point(candidates, r=half, p=np.inf, return_length=True)
    total_counts = all_tree.query_ball_point(candidates, r=half, p=np.inf, return_length=True)
    big_fraction = big_counts / np.maximum(total_counts, 1)

    eligible = np.ones(len(candidates), dtype=bool)
    if max_total_masks is not None:
        eligible &= total_counts <= int(max_total_masks)
    if not eligible.any():
        eligible = np.ones(len(candidates), dtype=bool)

    eligible_indices = np.flatnonzero(eligible)
    best_idx = max(
        eligible_indices,
        key=lambda i: (int(big_counts[i]), float(big_fraction[i]), -int(total_counts[i])),
    )
    cx, cy = candidates[best_idx]
    bbox = (float(cx - half), float(cy - half), float(cx + half), float(cy + half))
    scan_results = pd.DataFrame(
        {
            "center_x": candidates[:, 0],
            "center_y": candidates[:, 1],
            "big_mask_count": big_counts,
            "total_mask_count": total_counts,
            "big_mask_fraction": big_fraction,
        }
    ).sort_values(
        ["big_mask_count", "big_mask_fraction", "total_mask_count"],
        ascending=[False, False, True],
    )
    scan_summary = {
        "big_cutoff_um2": big_cutoff_um2,
        "big_cutoff_px": big_cutoff_px,
        "big_percentile": float(big_percentile),
        "chosen_big_count": float(big_counts[best_idx]),
        "chosen_total_count": float(total_counts[best_idx]),
        "chosen_big_fraction": float(big_fraction[best_idx]),
    }
    return bbox, scan_results, scan_summary


In [ ]:
big_crop_bbox, big_scan_results, big_scan_summary = scan_for_big_mask_crop(
    sdata,
    CELLPOSE_SHAPE_KEY,
    crop_size_um=BIG_CROP_SIZE_UM,
    big_percentile=BIG_MASK_PERCENTILE,
    big_min_area_um2=BIG_MASK_MIN_AREA_UM2,
    max_total_masks=BIG_SCAN_MAX_TOTAL_MASKS,
)
big_cellpose_crop = crop_shapes_with_areas(sdata, CELLPOSE_SHAPE_KEY, big_crop_bbox)
big_proseg_crop = crop_shapes_with_areas(sdata, PROSEG_SHAPE_KEY, big_crop_bbox)

bx0, by0, bx1, by1 = big_crop_bbox
print(f"Big-mask crop bbox: x=({bx0:.1f}, {bx1:.1f}), y=({by0:.1f}, {by1:.1f})")
print(
    f"Large cutoff >= "
    f"{big_scan_summary['big_cutoff_um2']:.3g} um2 / "
    f"{big_scan_summary['big_cutoff_px']:.1f} px-equivalent"
)
print(
    "Chosen window: "
    f"{big_scan_summary['chosen_big_count']:.0f} large masks, "
    f"{big_scan_summary['chosen_total_count']:.0f} total masks, "
    f"fraction={big_scan_summary['chosen_big_fraction']:.3f}"
)
big_scan_results.head(10)


In [ ]:
big_bg = _get_background_image_crop(sdata, DATASET_NAME, big_crop_bbox, zarr_path=ZARR_PATH) if SHOW_BACKGROUND else None
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
draw_mask_panel(
    axes[0],
    big_cellpose_crop,
    f"Cellpose big-mask crop ({len(big_cellpose_crop)})",
    "#a855f7",
    big_crop_bbox,
    bg=big_bg,
    label_all=True,
    low_area_label_cap=low_area_label_cap_current_mode,
)
draw_mask_panel(
    axes[1],
    big_proseg_crop,
    f"ProSeg big-mask crop ({len(big_proseg_crop)})",
    "#22c55e",
    big_crop_bbox,
    bg=big_bg,
    label_all=True,
    low_area_label_cap=low_area_label_cap_current_mode,
)
big_mask_plot_path = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_cellpose_proseg_large_area_crop.png"
fig.savefig(big_mask_plot_path, dpi=220, bbox_inches="tight")
print(f"Saved {big_mask_plot_path}")
plt.show()


In [ ]:
area_col = _area_mode_col()
areas = cellpose_areas[area_col].to_numpy(dtype=float)
areas = areas[np.isfinite(areas) & (areas > 0)]
big_hist_min = np.percentile(areas, BIG_MASK_PERCENTILE)
top_areas = areas[areas >= big_hist_min]
big_min_area_current_mode = BIG_MASK_MIN_AREA_UM2 if AREA_MODE == "um2" else BIG_MASK_MIN_AREA_UM2 / PIXEL_AREA_UM2

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
ax.hist(top_areas, bins=BIG_HIST_BINS, color="#2563eb", alpha=0.78)
for pct, color in [(BIG_MASK_PERCENTILE, "#dc2626"), (99.5, "#f97316"), (99.9, "#111827")]:
    if pct >= BIG_MASK_PERCENTILE:
        value = np.percentile(areas, pct)
        ax.axvline(value, color=color, linewidth=1.4, linestyle="--", label=f"p{pct:g}: {value:.1f}")
if big_min_area_current_mode >= big_hist_min:
    ax.axvline(
        big_min_area_current_mode,
        color="#16a34a",
        linewidth=2.0,
        linestyle="-",
        label=f">= {BIG_MASK_MIN_AREA_UM2:g} um2",
    )
ax.set_title(f"P7513 Cellpose mask areas, top {100 - BIG_MASK_PERCENTILE:g}%")
ax.set_xlabel(_area_axis_label())
ax.set_ylabel("mask count")
ax.legend(frameon=False, fontsize=8)

big_hist_path = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_cellpose_large_area_histogram.png"
fig.savefig(big_hist_path, dpi=220, bbox_inches="tight")
print(f"Saved {big_hist_path}")
plt.show()


## Percentiles

In [ ]:
percentiles = np.array([0, 0.5, 1, 2, 5, 10, 25, 50, 75, 90, 95, 99, 99.5, 100], dtype=float)
percentile_table = pd.DataFrame(
    {
        "percentile": percentiles,
        "area_um2": np.percentile(cellpose_areas["area_um2"].to_numpy(dtype=float), percentiles),
        "area_px": np.percentile(cellpose_areas["area_px"].to_numpy(dtype=float), percentiles),
    }
)
percentile_table

Set `AREA_CUTOFF` in the settings cell, rerun the area table, mask comparison, and histogram cells, and Cellpose masks below that cutoff will be highlighted in red in the zoomed Cellpose panel.